# PSASE Lab

# Experiment 2: Multiple Linear Regression

This is our Experiment 2, we will implement **Multiple Linear Regression** to predict house prices using multiple features (size, bedrooms, floors, age).


# Outline
- [ 1 - Packages ](#1)
- [ 2 - Problem Statement ](#2)
- [ 3 - Dataset ](#3)
  - [ 3.1 View the variables ](#3.1)
  - [ 3.2 Visualize the data ](#3.2)
- [ 4 - Multiple Linear Regression Model ](#4)
  - [ 4.1 Notation ](#4.1)
  - [ 4.2 Single Prediction (element by element) ](#4.2)
  - [ 4.3 Single Prediction (vectorized using np.dot) ](#4.3)
- [ 5 - Compute Cost ](#5)
- [ 6 - Gradient Descent ](#6)
  - [ 6.1 Compute Gradient ](#6.1)
  - [ 6.2 Gradient Descent Implementation ](#6.2)
- [ 7 - Learning Parameters using Batch Gradient Descent ](#7)
- [ 8 - Predictions ](#8)

<a name="1"></a>
## 1 - Packages

First, let's run the cell below to import all the packages that we will need during this experiment.
- [numpy](www.numpy.org) is the fundamental package for working with matrices in Python.
- [matplotlib](http://matplotlib.org) is a famous library to plot graphs in Python.
- [plotly](https://plotly.com/python/) is used for interactive 3D visualizations.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import copy
import math
%matplotlib inline

np.set_printoptions(precision=2)

<a name="2"></a>
## 2 - Problem Statement

Suppose you are a real estate agent and you want to predict the price of houses based on their features.

You have collected data on houses that were recently sold, and you have the following features for each house:

| Feature | Description | Units |
|:--------|:------------|:------|
| Size | Area of the house | sqft |
| Bedrooms | Number of bedrooms | count |
| Floors | Number of floors | count |
| Age | Age of the house | years |
| **Price** | **Sale price (target)** | **1000s of dollars** |

Can you use the data to help you predict the price for new houses based on their features?

Unlike simple linear regression where we had just one feature (e.g., population → profit), here we have **multiple features** ($n = 4$), so we need **multiple linear regression**.

## 3 - Dataset

Let's load the dataset from `data/houses.txt`. The file contains 200 examples with 4 features and 1 target variable.

In [ ]:
# Load the dataset
data = np.loadtxt('data/houses.txt', delimiter=',')
X_train = data[:, :4]   # Features: size, bedrooms, floors, age
y_train = data[:, 4]    # Target: price (in 1000s of dollars)

print(f"Dataset loaded successfully with {X_train.shape[0]} examples.")

<a name="3.1"></a>
### 3.1 View the variables

Before starting on any task, it is useful to get more familiar with the dataset.  
A good place to start is to just print out each variable and see what it contains.

Note that unlike simple linear regression, our feature matrix $\mathbf{X}$ is now a **2D array** with shape $(m, n)$ where:
- $m$ = number of training examples
- $n$ = number of features

In [ ]:
# Print X_train
print("Type of X_train:", type(X_train))
print("Shape of X_train:", X_train.shape)
print("First five rows of X_train:\n", X_train[:5])

Each row of `X_train` is a vector $\mathbf{x}^{(i)}$ containing the 4 features for the $i^{th}$ house:
- Column 0: Size (sqft)
- Column 1: Bedrooms
- Column 2: Floors
- Column 3: Age (years)

Now, let's print `y_train`:

In [ ]:
# Print y_train
print("Type of y_train:", type(y_train))
print("Shape of y_train:", y_train.shape)
print("First five elements of y_train:\n", y_train[:5])

In [ ]:
# Dimensions
print('The shape of X_train is:', X_train.shape)
print('The shape of y_train is:', y_train.shape)
print('Number of training examples (m):', X_train.shape[0])
print('Number of features (n):', X_train.shape[1])

<a name="3.2"></a>
### 3.2 Visualize the data

With multiple features, we can create scatter plots for each feature vs. the target (price) to understand the relationships.

In [ ]:
feature_names = ['Size (sqft)', 'Bedrooms', 'Floors', 'Age (years)']

fig, axes = plt.subplots(1, 4, figsize=(20, 4), sharey=True)
for i in range(4):
    axes[i].scatter(X_train[:, i], y_train, marker='x', c='r')
    axes[i].set_xlabel(feature_names[i])
    if i == 0:
        axes[i].set_ylabel('Price (in $1000s)')
    axes[i].set_title(f'Price vs {feature_names[i]}')
plt.tight_layout()
plt.show()

Our goal is to build a **multiple linear regression** model to fit this data.
- With this model, you can then input a new house's features (size, bedrooms, floors, age), and have the model estimate its price.

<a name="4"></a>
## 4 - Multiple Linear Regression Model

<a name="4.1"></a>
### 4.1 Notation

Here is a summary of the notation used in multiple linear regression:

| General Notation | Description | Python (if applicable) |
|:----------------:|:-----------:|:----------------------:|
|      $a$         | scalar, non-bold | |
|  $\mathbf{a}$   | vector, bold | |
|  $\mathbf{A}$   | matrix, bold capital | |
| **Regression** | | |
|  $\mathbf{X}$   | training example matrix | `X_train` |
|  $\mathbf{y}$   | training example targets | `y_train` |
| $\mathbf{x}^{(i)}$, $y^{(i)}$ | $i_{th}$ Training Example | `X[i]`, `y[i]` |
|        $m$       | number of training examples | `m` |
|        $n$       | number of features in each example | `n` |
|  $\mathbf{w}$   | parameter: weight vector | `w` |
|        $b$       | parameter: bias | `b` |
| $f_{\mathbf{w},b}(\mathbf{x}^{(i)})$ | Model prediction: $\mathbf{w} \cdot \mathbf{x}^{(i)}+b$ | `f_wb` |

The model's prediction with multiple variables is given by the linear model:

$$ f_{\mathbf{w},b}(\mathbf{x}) =  w_0x_0 + w_1x_1 +... + w_{n-1}x_{n-1} + b \tag{1}$$

or in **vector notation**:

$$ f_{\mathbf{w},b}(\mathbf{x}) = \mathbf{w} \cdot \mathbf{x} + b  \tag{2} $$ 

where $\cdot$ is a vector `dot product`.

The key difference from simple linear regression:
- $\mathbf{w}$ is now a **vector** with $n$ elements (one weight per feature), not a scalar.
- $\mathbf{x}^{(i)}$ is also a **vector** with $n$ elements.

<a name="4.2"></a>
### 4.2 Single Prediction (element by element)

Our previous single-variable prediction multiplied one feature value by one parameter and added a bias. 

A direct extension to multiple features would be to implement equation (1) using a loop over each element:

In [ ]:
def predict_single_loop(x, w, b): 
    """
    Single prediction using linear regression (loop version)
    
    Args:
      x (ndarray): Shape (n,) example with multiple features
      w (ndarray): Shape (n,) model parameters    
      b (scalar):  model parameter     
      
    Returns:
      p (scalar):  prediction
    """
    n = x.shape[0]
    p = 0
    for i in range(n):
        p_i = x[i] * w[i]  
        p = p + p_i         
    p = p + b                
    return p

In [ ]:
# Test with some initial parameters
# These are near-optimal values for demonstration
b_init = 785.1811367994083
w_init = np.array([ 0.39133535, 18.75376741, -53.36032453, -26.42131618])
print(f"w_init shape: {w_init.shape}, b_init type: {type(b_init)}")

# Get a row from our training data
x_vec = X_train[0, :]
print(f"\nx_vec shape {x_vec.shape}, x_vec value: {x_vec}")

# Make a prediction
f_wb = predict_single_loop(x_vec, w_init, b_init)
print(f"f_wb prediction: {f_wb:.4f}")
print(f"Actual value: {y_train[0]}")

<a name="4.3"></a>
### 4.3 Single Prediction (vectorized using np.dot)

Equation (1) can be implemented much more efficiently using the **dot product** as in equation (2).

NumPy `np.dot()` performs a vector dot product, which is both faster and more concise:

In [ ]:
def predict(x, w, b): 
    """
    Single prediction using linear regression (vectorized)
    Args:
      x (ndarray): Shape (n,) example with multiple features
      w (ndarray): Shape (n,) model parameters   
      b (scalar):             model parameter 
      
    Returns:
      p (scalar):  prediction
    """
    p = np.dot(x, w) + b     
    return p    

In [ ]:
# Verify: both methods should give the same result
x_vec = X_train[0, :]
f_wb_loop = predict_single_loop(x_vec, w_init, b_init)
f_wb_vec = predict(x_vec, w_init, b_init)

print(f"Loop prediction: {f_wb_loop:.4f}")
print(f"Vectorized prediction: {f_wb_vec:.4f}")
print(f"Both methods match: {np.isclose(f_wb_loop, f_wb_vec)}")

The results are the same! Going forward, we will use `np.dot` for efficiency. The prediction is now a **single statement** instead of a loop.

<a name="5"></a>
## 5 - Compute Cost

Gradient descent involves repeated steps to adjust the value of your parameters $(\mathbf{w}, b)$ to gradually get a smaller and smaller cost $J(\mathbf{w},b)$.

The cost function for multiple linear regression $J(\mathbf{w},b)$ is defined as:

$$J(\mathbf{w},b) = \frac{1}{2m} \sum\limits_{i = 0}^{m-1} (f_{\mathbf{w},b}(\mathbf{x}^{(i)}) - y^{(i)})^2 \tag{3}$$

where:

$$ f_{\mathbf{w},b}(\mathbf{x}^{(i)}) = \mathbf{w} \cdot \mathbf{x}^{(i)} + b  \tag{4} $$ 

Note that $\mathbf{w}$ and $\mathbf{x}^{(i)}$ are **vectors** rather than scalars, supporting multiple features.

In [ ]:
def compute_cost(X, y, w, b): 
    """
    Computes the cost function for multiple linear regression.

    Args:
        X (ndarray (m,n)): Data, m examples with n features
        y (ndarray (m,)) : target values
        w (ndarray (n,)) : model parameters  
        b (scalar)       : model parameter

    Returns:
        total_cost (float): The cost of using w,b as the parameters for
                            linear regression to fit the data
    """
    m = X.shape[0]
    cost = 0.0
    for i in range(m):                                
        f_wb_i = np.dot(X[i], w) + b
        cost = cost + (f_wb_i - y[i])**2
    total_cost = cost / (2 * m)
    return total_cost

In [ ]:
# Compute cost with initial w and b = 0
initial_w = np.zeros(X_train.shape[1])
initial_b = 0

cost = compute_cost(X_train, y_train, initial_w, initial_b)
print(f'Cost at initial w (zeros), b (0): {cost:.3f}')

# Compute cost with near-optimal parameters
cost_opt = compute_cost(X_train, y_train, w_init, b_init)
print(f'Cost at near-optimal w, b: {cost_opt:.6e}')

### 3D Visualization of Cost Function (Soup Bowl)

Let's visualize the cost function as a 3D surface. Since we have 4 weights + 1 bias, we can't visualize all dimensions at once. Instead, we'll create a simplified bowl shape to illustrate the concept of the cost surface.

In [ ]:
import plotly.graph_objects as go

def soup_bowl():
    """ Create figure and plot with a 3D projection"""

    w = np.linspace(-20, 20, 100)
    b = np.linspace(-20, 20, 100)

    z = np.zeros((len(w), len(b)))
    j = 0
    for x in w:
        i = 0
        for y in b:
            z[i, j] = x**2 + y**2
            i += 1
        j += 1

    W, B = np.meshgrid(w, b)

    fig = go.Figure(data=[
        go.Surface(z=z, x=W, y=B, colorscale='Spectral_r', opacity=0.7,
                   contours=dict(z=dict(show=False))),
    ])

    fig.update_layout(
        title=dict(text="<b>J(w,b)</b><br>[You can rotate this figure]", x=0.5, font=dict(size=15)),
        scene=dict(
            xaxis=dict(title="w", backgroundcolor="white", gridcolor="lightgrey"),
            yaxis=dict(title="b", backgroundcolor="white", gridcolor="lightgrey"),
            zaxis=dict(title="J(w,b)", backgroundcolor="white", gridcolor="lightgrey"),
            camera=dict(eye=dict(x=1.5, y=-1.5, z=1.2))
        ),
        width=800,
        height=600,
    )

    fig.show()

soup_bowl()

This 3D plot shows the "soup bowl" shape of a cost function. The goal of gradient descent is to find the bottom of this bowl — the values of $\mathbf{w}$ and $b$ that minimize the cost.

<a name="6"></a>
## 6 - Gradient Descent

The gradient descent algorithm for **multiple variables** is:

$$\begin{align*} \text{repeat}& \text{ until convergence:} \; \lbrace \newline\;
& w_j = w_j -  \alpha \frac{\partial J(\mathbf{w},b)}{\partial w_j} \tag{5}  \; & \text{for j = 0..n-1}\newline
&b\ \ = b -  \alpha \frac{\partial J(\mathbf{w},b)}{\partial b}  \newline \rbrace
\end{align*}$$

where, $n$ is the number of features, parameters $w_j$,  $b$, are **updated simultaneously** and where  

$$
\begin{align}
\frac{\partial J(\mathbf{w},b)}{\partial w_j}  &= \frac{1}{m} \sum\limits_{i = 0}^{m-1} (f_{\mathbf{w},b}(\mathbf{x}^{(i)}) - y^{(i)})x_{j}^{(i)} \tag{6}  \\
\frac{\partial J(\mathbf{w},b)}{\partial b}  &= \frac{1}{m} \sum\limits_{i = 0}^{m-1} (f_{\mathbf{w},b}(\mathbf{x}^{(i)}) - y^{(i)}) \tag{7}
\end{align}
$$

* $m$ is the number of training examples in the data set
* $f_{\mathbf{w},b}(\mathbf{x}^{(i)})$ is the model's prediction, while $y^{(i)}$ is the target value

<a name="6.1"></a>
### 6.1 Compute Gradient

The implementation below calculates equations (6) and (7) using:
- An outer loop over all $m$ examples
  - $\frac{\partial J}{\partial b}$ is accumulated directly
  - An inner loop over all $n$ features computes $\frac{\partial J}{\partial w_j}$

In [ ]:
def compute_gradient(X, y, w, b): 
    """
    Computes the gradient for linear regression 
    Args:
      X (ndarray (m,n)): Data, m examples with n features
      y (ndarray (m,)) : target values
      w (ndarray (n,)) : model parameters  
      b (scalar)       : model parameter
      
    Returns:
      dj_dw (ndarray (n,)): The gradient of the cost w.r.t. the parameters w. 
      dj_db (scalar):       The gradient of the cost w.r.t. the parameter b. 
    """
    m, n = X.shape           # (number of examples, number of features)
    dj_dw = np.zeros((n,))
    dj_db = 0.

    for i in range(m):                             
        err = (np.dot(X[i], w) + b) - y[i]   
        for j in range(n):                         
            dj_dw[j] = dj_dw[j] + err * X[i, j]    
        dj_db = dj_db + err                        
    dj_dw = dj_dw / m                                
    dj_db = dj_db / m                                
        
    return dj_dw, dj_db

In [ ]:
# Compute and display gradient with w initialized to zeroes
initial_w = np.zeros(X_train.shape[1])
initial_b = 0

tmp_dj_dw, tmp_dj_db = compute_gradient(X_train, y_train, initial_w, initial_b)
print('Gradient at initial w (zeros), b (0):')
print(f'  dj_dw: {tmp_dj_dw}')
print(f'  dj_db: {tmp_dj_db:.4f}')

<a name="6.2"></a>
### 6.2 Gradient Descent Implementation

Now let's put it all together. The `gradient_descent` function implements equation (5) above.

In [ ]:
def gradient_descent(X, y, w_in, b_in, cost_function, gradient_function, alpha, num_iters): 
    """
    Performs batch gradient descent to learn w and b. Updates w and b by taking 
    num_iters gradient steps with learning rate alpha
    
    Args:
      X (ndarray (m,n))   : Data, m examples with n features
      y (ndarray (m,))    : target values
      w_in (ndarray (n,)) : initial model parameters  
      b_in (scalar)       : initial model parameter
      cost_function       : function to compute cost
      gradient_function   : function to compute the gradient
      alpha (float)       : Learning rate
      num_iters (int)     : number of iterations to run gradient descent
      
    Returns:
      w (ndarray (n,)) : Updated values of parameters 
      b (scalar)       : Updated value of parameter 
    """
    
    # An array to store cost J at each iteration primarily for graphing later
    J_history = []
    w = copy.deepcopy(w_in)  # avoid modifying global w within function
    b = b_in
    
    for i in range(num_iters):

        # Calculate the gradient and update the parameters
        dj_dw, dj_db = gradient_function(X, y, w, b)

        # Update Parameters using w, b, alpha and gradient
        w = w - alpha * dj_dw
        b = b - alpha * dj_db
      
        # Save cost J at each iteration
        if i < 100000:      # prevent resource exhaustion 
            J_history.append(cost_function(X, y, w, b))

        # Print cost every at intervals 10 times or as many iterations if < 10
        if i % math.ceil(num_iters / 10) == 0:
            print(f"Iteration {i:4d}: Cost {J_history[-1]:8.2f}   ")
        
    return w, b, J_history  # return final w,b and J history for graphing

<a name="7"></a>
## 7 - Learning Parameters using Batch Gradient Descent

Now let's run gradient descent on our dataset to learn the optimal parameters.

**Note**: With raw features, the learning rate must be kept very small. This is because the features have very different scales (size ranges from ~600-3500 sqft while floors is just 1-2). Feature scaling (covered in a later lab) would fix this.

In [ ]:
# Initialize parameters
initial_w = np.zeros(X_train.shape[1])
initial_b = 0.

# Some gradient descent settings
iterations = 1000
alpha = 5.0e-7

# Run gradient descent 
w_final, b_final, J_hist = gradient_descent(X_train, y_train, initial_w, initial_b,
                                            compute_cost, compute_gradient, 
                                            alpha, iterations)
print(f"\nb,w found by gradient descent: b = {b_final:0.2f}, w = {w_final}")

In [ ]:
# Plot cost versus iteration to see convergence
fig, (ax1, ax2) = plt.subplots(1, 2, constrained_layout=True, figsize=(12, 4))
ax1.plot(J_hist)
ax2.plot(100 + np.arange(len(J_hist[100:])), J_hist[100:])
ax1.set_title("Cost vs. iteration");  ax2.set_title("Cost vs. iteration (tail)")
ax1.set_ylabel('Cost')             ;  ax2.set_ylabel('Cost') 
ax1.set_xlabel('iteration step')   ;  ax2.set_xlabel('iteration step') 
plt.show()

In [ ]:
# Print predictions vs actuals for the first few training examples
print("Predictions vs Actual values (first 10 examples):")
print(f"{'Example':>8}  {'Prediction':>12}  {'Actual':>10}")
print("-" * 35)
for i in range(min(10, X_train.shape[0])):
    pred = np.dot(X_train[i], w_final) + b_final
    print(f"{i:>8}  {pred:>12.2f}  {y_train[i]:>10.2f}")

**Note**: The predictions might not be very accurate yet because:
1. The features have very different scales (size in 100s-1000s, floors is 1-2), which slows convergence
2. We used a very small learning rate to avoid divergence

Feature normalization (z-score normalization) would significantly improve gradient descent performance. This is covered in a later experiment.

<a name="8"></a>
## 8 - Predictions

Now we can use our trained model to make predictions on new houses. Let's predict the price for some sample houses using the learned parameters $\mathbf{w}$ and $b$.

In [ ]:
# Prediction 1: A 1200 sqft house, 3 bedrooms, 1 floor, 40 years old
x_test1 = np.array([1200, 3, 1, 40])
predict1 = np.dot(x_test1, w_final) + b_final
print('For a 1200 sqft, 3 bed, 1 floor, 40 yr old house, we predict a price of $%.2f' % (predict1 * 1000))

# Prediction 2: A 2000 sqft house, 4 bedrooms, 2 floors, 10 years old
x_test2 = np.array([2000, 4, 2, 10])
predict2 = np.dot(x_test2, w_final) + b_final
print('For a 2000 sqft, 4 bed, 2 floor, 10 yr old house, we predict a price of $%.2f' % (predict2 * 1000))

# Prediction 3: A 800 sqft house, 2 bedrooms, 1 floor, 70 years old
x_test3 = np.array([800, 2, 1, 70])
predict3 = np.dot(x_test3, w_final) + b_final
print('For a 800 sqft, 2 bed, 1 floor, 70 yr old house, we predict a price of $%.2f' % (predict3 * 1000))

# Prediction 4: A 3000 sqft house, 4 bedrooms, 2 floors, 5 years old
x_test4 = np.array([3000, 4, 2, 5])
predict4 = np.dot(x_test4, w_final) + b_final
print('For a 3000 sqft, 4 bed, 2 floor, 5 yr old house, we predict a price of $%.2f' % (predict4 * 1000))

## Congratulations!

In this experiment you:
- Extended the data structures and routines from simple linear regression to support **multiple features**
- Loaded a **real dataset** with 200 training examples and 4 features
- Implemented prediction both **element by element** (loop) and **vectorized** (using `np.dot`)
- Implemented the **cost function** for multiple linear regression
- Implemented **gradient computation** with multiple variables 
- Implemented **gradient descent** to learn optimal parameters
- Used the learned parameters to **predict house prices** for new houses
- Utilized NumPy `np.dot` to vectorize implementations for **speed and simplicity**